#Task 8: Content-Based Filtering with a Neural Network

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Concatenate, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# 1. Load Data
r_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv('u.data', sep='\t', names=r_cols, encoding='latin-1')

m_cols = ['movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url'] + \
         ['unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy',
          'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
          'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
movies = pd.read_csv('u.item', sep='|', names=m_cols, encoding='latin-1')

# 2. Extract Release Year
movies['release_year'] = pd.to_datetime(movies['release_date'], errors='coerce').dt.year
movies['release_year'] = movies['release_year'].fillna(movies['release_year'].median())

# 3. Train/Test Split
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)
global_avg = train_data['rating'].mean()

# 4. Movie Features: Calculate Average Rating per Movie
movie_avgs = train_data.groupby('movie_id')['rating'].mean().rename('movie_avg_rating')
movies = movies.merge(movie_avgs, on='movie_id', how='left')
movies['movie_avg_rating'] = movies['movie_avg_rating'].fillna(global_avg)

# 5. User Features: Calculate Average Rating per Genre for each User
train_merged = train_data.merge(movies, on='movie_id')
genre_cols = movies.columns[5:24] # The 19 genre columns

user_genre_avgs = {}
for genre in genre_cols:
    # Filter where the movie belongs to the genre, then group by user and get mean rating
    genre_specific_ratings = train_merged[train_merged[genre] == 1]
    user_genre_avgs[genre] = genre_specific_ratings.groupby('user_id')['rating'].mean()

user_features_df = pd.DataFrame(user_genre_avgs).fillna(global_avg)

print("Feature engineering complete!")

Feature engineering complete!


In [2]:
# 1. Scale continuous movie features (Year and Avg Rating)
scaler = StandardScaler()
movies[['release_year', 'movie_avg_rating']] = scaler.fit_transform(movies[['release_year', 'movie_avg_rating']])

# 2. Build the exact feature arrays mapping to the train and test sets
def generate_nn_inputs(df):
    user_inputs = []
    movie_inputs = []
    targets = []

    for _, row in df.iterrows():
        u_id = row['user_id']
        m_id = row['movie_id']

        # Get User Vector (19 genre averages)
        if u_id in user_features_df.index:
            u_vec = user_features_df.loc[u_id].values
        else:
            u_vec = np.full(len(genre_cols), global_avg) # Cold start fallback

        # Get Movie Vector (19 genres + 1 year + 1 avg rating = 21 features)
        m_row = movies[movies['movie_id'] == m_id].iloc[0]
        m_vec = np.concatenate([m_row[genre_cols].values, [m_row['release_year'], m_row['movie_avg_rating']]])

        user_inputs.append(u_vec)
        movie_inputs.append(m_vec)
        targets.append(row['rating'])

    return np.array(user_inputs, dtype='float32'), np.array(movie_inputs, dtype='float32'), np.array(targets, dtype='float32')

print("Formatting training data...")
X_train_user, X_train_movie, y_train = generate_nn_inputs(train_data)

print("Formatting testing data...")
X_test_user, X_test_movie, y_test = generate_nn_inputs(test_data)

print(f"User Input Shape: {X_train_user.shape}")
print(f"Movie Input Shape: {X_train_movie.shape}")

Formatting training data...
Formatting testing data...
User Input Shape: (80000, 19)
Movie Input Shape: (80000, 21)


In [3]:
user_input = Input(shape=(19,), name='User_Features')
movie_input = Input(shape=(21,), name='Movie_Features')

user_branch = Dense(32, activation='relu')(user_input)
user_branch = Dense(16, activation='relu')(user_branch)

movie_branch = Dense(32, activation='relu')(movie_input)
movie_branch = Dense(16, activation='relu')(movie_branch)

merged = Concatenate()([user_branch, movie_branch])

dense_1 = Dense(32, activation='relu')(merged)
dropout = Dropout(0.2)(dense_1)
output = Dense(1, activation='linear', name='Rating_Prediction')(dropout)

model = Model(inputs=[user_input, movie_input], outputs=output)
custom_adam = Adam(learning_rate=0.001)
model.compile(optimizer=custom_adam, loss='mse')

model.summary()

print("\nTraining the Neural Network...")
history = model.fit(
    [X_train_user, X_train_movie], y_train,
    validation_data=([X_test_user, X_test_movie], y_test),
    batch_size=256,
    epochs=15,
    verbose=1
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ User_Features       │ (None, 19)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Movie_Features      │ (None, 21)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │        640 │ User_Features[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │        704 │ Movie_Features[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │        528 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 16)        │        528 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 32)        │          0 │ dense_1[0][0],    │
│ (Concatenate)       │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 32)        │      1,056 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 32)        │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Rating_Prediction   │ (None, 1)         │         33 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,489 (13.63 KB)

 Trainable params: 3,489 (13.63 KB)

 Non-trainable params: 0 (0.00 B)


Training the Neural Network...
Epoch 1/15
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 2.7910 - val_loss: 0.9301
Epoch 2/15
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.1432 - val_loss: 0.9246
Epoch 3/15
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 1.1131 - val_loss: 0.9246
Epoch 4/15
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 1.1010 - val_loss: 0.9266
Epoch 5/15
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1.0975 - val_loss: 0.9131
Epoch 6/15
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 1.0871 - val_loss: 0.9178
Epoch 7/15
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.0722 - val_loss: 0.9111
Epoch 8/15
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.0644 - val_loss: 0.9232
Epoch 9/15
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.0594 - val_loss: 0.9292
Epoch 10/15
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.0472 - val_loss: 0.9163
Epoch 11/15
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 1.0346 - val_loss: 0.9338
Epoch 12/15
313/313 

In [4]:
nn_predictions = model.predict([X_test_user, X_test_movie]).flatten()
nn_predictions = np.clip(nn_predictions, 1, 5) # Ratings must be between 1 and 5

nn_mse = mean_squared_error(y_test, nn_predictions)
nn_rmse = np.sqrt(nn_mse)

print("--- Task 8: Deep Learning CBF Evaluation ---")
print(f"Neural Network MSE:  {nn_mse:.4f}")
print(f"Neural Network RMSE: {nn_rmse:.4f}")

625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
--- Task 8: Deep Learning CBF Evaluation ---
Neural Network MSE:  0.9126
Neural Network RMSE: 0.9553


#Task 9: Reinforcement Learning in Recommender Systems

In [5]:
import pandas as pd
import numpy as np
import random
import math

r_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv('u.data', sep='\t', names=r_cols, encoding='latin-1')

users = ratings['user_id'].unique()
movies = ratings['movie_id'].unique()
num_users = len(users)
num_movies = len(movies)

user_to_idx = {u: i for i, u in enumerate(users)}
movie_to_idx = {m: i for i, m in enumerate(movies)}

reward_matrix = np.zeros((num_users, num_movies))

for _, row in ratings.iterrows():
    u_idx = user_to_idx[row['user_id']]
    m_idx = movie_to_idx[row['movie_id']]
    rating = row['rating']

    if rating >= 4.0:
        reward_matrix[u_idx, m_idx] = 1.0
    else:
        reward_matrix[u_idx, m_idx] = -1.0

print(f"Environment initialized. State Space: {num_users} Users, Action Space: {num_movies} Movies.")

Environment initialized. State Space: 943 Users, Action Space: 1682 Movies.


In [6]:
class MultiArmedBandit:
    def __init__(self, num_actions):
        self.num_actions = num_actions
        self.q_values = np.zeros(num_actions)  # Estimated rewards
        self.action_counts = np.zeros(num_actions) # How many times an action was taken

    def select_action_epsilon_greedy(self, epsilon=0.1):
        """Explores randomly with probability epsilon, otherwise exploits best known action."""
        if np.random.rand() < epsilon:
            return np.random.randint(self.num_actions)
        else:
            return np.argmax(self.q_values)

    def select_action_ucb(self, t, c=1.0):
        """Uses Upper Confidence Bound to balance exploration and exploitation."""

        unvisited = np.where(self.action_counts == 0)[0]
        if len(unvisited) > 0:
            return np.random.choice(unvisited)

        # UCB Formula: Q(a) + c * sqrt(ln(t) / N(a))
        confidence_bounds = self.q_values + c * np.sqrt(np.log(t) / self.action_counts)
        return np.argmax(confidence_bounds)

    def update(self, action, reward):
        """Updates the reward estimate for the chosen action."""
        self.action_counts[action] += 1
        # Incremental average update
        self.q_values[action] += (1 / self.action_counts[action]) * (reward - self.q_values[action])

# Simulate MAB Training
epochs = 10000
mab_agent = MultiArmedBandit(num_movies)

for t in range(1, epochs + 1):

    user_idx = np.random.randint(num_users)

    action_idx = mab_agent.select_action_ucb(t, c=1.5)

    reward = reward_matrix[user_idx, action_idx]

    mab_agent.update(action_idx, reward)

print("MAB Training complete. Top 5 Global Movies (Indices) learned:")
print(np.argsort(mab_agent.q_values)[-5:][::-1])

MAB Training complete. Top 5 Global Movies (Indices) learned:
[ 31 299 161 200 216]


In [7]:
# Initialize Q-Table with small random values
q_table = np.random.uniform(low=-0.01, high=0.01, size=(num_users, num_movies))

# Hyperparameters
alpha = 0.1      # Learning rate
gamma = 0.9      # Discount factor
epsilon = 0.2    # Exploration rate
episodes = 50000 # Number of interactions

print("Training Q-Learning Agent...")

for episode in range(episodes):
    s = np.random.randint(num_users)
    if np.random.rand() < epsilon:
        a = np.random.randint(num_movies) # Explore
    else:
        a = np.argmax(q_table[s])         # Exploit
    r = reward_matrix[s, a]
    s_prime = s

    best_next_action = np.argmax(q_table[s_prime])
    td_target = r + gamma * q_table[s_prime, best_next_action]
    td_error = td_target - q_table[s, a]

    q_table[s, a] = q_table[s, a] + alpha * td_error

print("Q-Table Convergence complete!")

user_idx = 0
top_actions = np.argsort(q_table[user_idx])[-3:][::-1]
print(f"Top 3 Movie Indices recommended for User 0: {top_actions}")

Training Q-Learning Agent...
Q-Table Convergence complete!
Top 3 Movie Indices recommended for User 0: [ 92 542 239]
